# EV Charging Demand Prediction
# Feature Engineering

# Step 1 — Load Cleaned Dataset

In [1]:
import pandas as pd

df = pd.read_csv("Data_row/cleaned data/ChargingRecords_Cleaned.csv")

df.head()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,CalculateDuration,PreciseDuration,InvalidNegativeDuration,ZeroDurationRecoverable,ZeroDurationUnresolved,CleanedDuration
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:00,2022-09-15 23:59:00,185,20.36,185.0,185.183333,False,False,False,185.0
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:00,2022-09-14 21:31:00,90,10.19,90.0,89.983333,False,False,False,90.0
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:00,2022-09-14 19:54:00,60,6.78,60.0,59.983333,False,False,False,60.0
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:00,2022-09-30 00:16:00,344,37.65,344.0,343.850000,False,False,False,344.0
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:00,2022-09-26 00:30:00,300,33.81,300.0,299.983333,False,False,False,300.0


# Step 2 — Create Date and Time Features

In [2]:
df["StartDatetime"] = pd.to_datetime(df["StartDatetime"])

df["StartHour"] = df["StartDatetime"].dt.hour
df["DayOfWeek"] = df["StartDatetime"].dt.dayofweek
df["Month"] = df["StartDatetime"].dt.month
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

df[["StartHour", "DayOfWeek", "Month", "IsWeekend"]].head()

,StartHour,DayOfWeek,Month,IsWeekend
0,20,3,9,0
1,20,2,9,0
2,18,2,9,0
3,18,3,9,0
4,19,6,9,1


# Step 3 — Check Charger Type

In [3]:
df["ChargerType"].value_counts(dropna=False)

ChargerType
0    57896
1    14947
Name: count, dtype: int64

# Step 4 — One-Hot Encoding

In [4]:
df_encoded = pd.get_dummies(
    df,
    columns=["Location", "ChargerType"],
    dtype=int
)

df_encoded.head()

,UserID,ChargerID,ChargerCompany,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,...,Location_hotel,Location_market,Location_public area,Location_public institution,Location_public parking lot,Location_resort,Location_restaurant,Location_sightseeing,ChargerType_0,ChargerType_1
0,0,1,1,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:00,2022-09-15 23:59:00,185,...,1,0,0,0,0,0,0,0,1,0
1,0,1,1,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:00,2022-09-14 21:31:00,90,...,1,0,0,0,0,0,0,0,1,0
2,0,1,1,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:00,2022-09-14 19:54:00,60,...,1,0,0,0,0,0,0,0,1,0
3,0,1,1,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:00,2022-09-30 00:16:00,344,...,1,0,0,0,0,0,0,0,1,0
4,0,1,1,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:00,2022-09-26 00:30:00,300,...,1,0,0,0,0,0,0,0,1,0


# Step 5 — Select Features and Target

In [5]:
feature_columns = [
    "StartHour",
    "DayOfWeek",
    "Month",
    "IsWeekend"
]

feature_columns += [
    col for col in df_encoded.columns
    if col.startswith(("Location_", "ChargerType_"))
]

X = df_encoded[feature_columns]
y = df_encoded["Demand"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (72843, 20)
y Shape: (72843,)


# Step 7 — Save Feature-Engineered Dataset

In [6]:
import os

os.makedirs("Data_row/feature_engineered", exist_ok=True)

df_final = X.copy()
df_final["Demand"] = y

df_final.to_csv(
    "Data_row/feature_engineered/EV_Features.csv",
    index=False
)

print("Feature-engineered dataset saved!")
print("Dataset Shape:", df_final.shape)

Feature-engineered dataset saved!
Dataset Shape: (72843, 21)


# step 8 Add TimeOfDay Feature

In [7]:
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["TimeOfDay"] = df["StartHour"].apply(get_time_of_day)

df[["StartHour", "TimeOfDay"]].head()

,StartHour,TimeOfDay
0,20,Evening
1,20,Evening
2,18,Evening
3,18,Evening
4,19,Evening


# Step 9: Update Encoded Features

In [8]:
# Convert TimeOfDay into numerical columns
time_encoded = pd.get_dummies(
    df["TimeOfDay"],
    prefix="TimeOfDay",
    dtype=int
)

# Add these columns to our existing features
X = pd.concat([X, time_encoded], axis=1)

# Update the final dataset
df_final = X.copy()
df_final["Demand"] = y

# Save the updated dataset
df_final.to_csv(
    "Data_row/feature_engineered/EV_Features.csv",
    index=False
)

print("Updated dataset saved!")
print("Dataset Shape:", df_final.shape)

Updated dataset saved!
Dataset Shape: (72843, 25)
